In [2]:
import sqlite3
import pandas as pd


conn = sqlite3.connect(":memory:")
conn.execute("PRAGMA foreign_keys = ON;")

conn.executescript("""
CREATE TABLE hw2_customers (
    customer_id INTEGER PRIMARY KEY,
    signup_date TEXT NOT NULL,
    country TEXT NOT NULL
);

CREATE TABLE hw2_customer_features_offline (
    feature_id INTEGER PRIMARY KEY AUTOINCREMENT,
    customer_id INTEGER NOT NULL,
    orders_30d INTEGER NOT NULL CHECK (orders_30d >= 0),
    avg_order_value REAL NOT NULL CHECK (avg_order_value >= 0),
    segment TEXT NOT NULL,
    valid_from TEXT NOT NULL,
    valid_to TEXT,
    FOREIGN KEY (customer_id)
        REFERENCES hw2_customers(customer_id),
    CHECK (valid_to IS NULL OR valid_to > valid_from)
);

CREATE TABLE hw2_training_events (
    event_id INTEGER PRIMARY KEY AUTOINCREMENT,
    customer_id INTEGER NOT NULL,
    prediction_ts TEXT NOT NULL,
    label_horizon_end TEXT NOT NULL,
    churn_in_horizon INTEGER NOT NULL,
    FOREIGN KEY (customer_id)
        REFERENCES hw2_customers(customer_id),
    CHECK (label_horizon_end > prediction_ts)
);
""")

print("Tables created.")


conn.executemany("""
INSERT INTO hw2_customers
(customer_id, signup_date, country)
VALUES (?, ?, ?)
""", [
    (1, "2024-01-10", "UA"),
    (2, "2024-02-20", "PL"),
    (42, "2024-03-15", "UA")
])


features = [
    (1, 1, 80.00, "A", "2024-10-01 00:00:00+00:00",
     "2024-11-01 00:00:00+00:00"),
    (1, 3, 100.00, "A", "2024-11-01 00:00:00+00:00",
     "2024-12-01 00:00:00+00:00"),
    (1, 5, 120.00, "B", "2024-12-01 00:00:00+00:00",
     "2025-01-01 00:00:00+00:00"),
    (1, 7, 150.00, "B", "2025-01-01 00:00:00+00:00",
     None),

    (2, 2, 50.00, "A", "2024-10-01 00:00:00+00:00",
     "2024-11-15 00:00:00+00:00"),
    (2, 4, 65.00, "A", "2024-11-15 00:00:00+00:00",
     "2024-12-15 00:00:00+00:00"),
    (2, 6, 90.00, "B", "2024-12-15 00:00:00+00:00",
     "2025-01-15 00:00:00+00:00"),
    (2, 8, 110.00, "C", "2025-01-15 00:00:00+00:00",
     None),

    (42, 1, 80.00, "A", "2024-10-01 00:00:00+00:00",
     "2024-11-01 00:00:00+00:00"),
    (42, 2, 100.00, "A", "2024-11-01 00:00:00+00:00",
     "2024-12-15 00:00:00+00:00"),
    (42, 5, 200.00, "B", "2024-12-15 00:00:00+00:00",
     "2025-01-15 00:00:00+00:00"),
    (42, 9, 250.00, "C", "2025-01-15 00:00:00+00:00",
     None)
]

conn.executemany("""
INSERT INTO hw2_customer_features_offline
(customer_id, orders_30d, avg_order_value, segment,
 valid_from, valid_to)
VALUES (?, ?, ?, ?, ?, ?)
""", features)

print("Added historical versions:", len(features))


events = [
    (1, "2024-10-20 10:00:00+00:00",
     "2024-11-19 10:00:00+00:00", 0),
    (1, "2024-11-20 10:00:00+00:00",
     "2024-12-20 10:00:00+00:00", 1),
    (1, "2025-01-10 10:00:00+00:00",
     "2025-02-09 10:00:00+00:00", 0),

    (2, "2024-10-20 10:00:00+00:00",
     "2024-11-19 10:00:00+00:00", 0),
    (2, "2024-12-20 10:00:00+00:00",
     "2025-01-19 10:00:00+00:00", 1),
    (2, "2025-02-01 10:00:00+00:00",
     "2025-03-03 10:00:00+00:00", 0),

    (42, "2024-10-20 10:00:00+00:00",
     "2024-11-19 10:00:00+00:00", 0),
    (42, "2024-12-10 10:00:00+00:00",
     "2025-01-09 10:00:00+00:00", 0),
    (42, "2024-12-20 10:00:00+00:00",
     "2025-01-19 10:00:00+00:00", 1),
    (42, "2025-01-20 10:00:00+00:00",
     "2025-02-19 10:00:00+00:00", 0)
]

conn.executemany("""
INSERT INTO hw2_training_events
(customer_id, prediction_ts, label_horizon_end, churn_in_horizon)
VALUES (?, ?, ?, ?)
""", events)

print("Educational events added:", len(events))


def show_query(title, query, params=()):
    print("\n" + "=" * 75)
    print(title)
    print("=" * 75)
    df = pd.read_sql_query(query, conn, params=params)
    display(df)
    return df


show_query(
    "AS-OF SELECT №1: client 42 on 10.12.2024",
    """
    SELECT customer_id, orders_30d, avg_order_value,
           segment, valid_from, valid_to
    FROM hw2_customer_features_offline
    WHERE customer_id = ?
      AND valid_from <= ?
      AND (valid_to > ? OR valid_to IS NULL)
    """,
    (
        42,
        "2024-12-10 10:00:00+00:00",
        "2024-12-10 10:00:00+00:00"
    )
)


show_query(
    "AS-OF SELECT №2: client 42 on 20.12.2024",
    """
    SELECT customer_id, orders_30d, avg_order_value,
           segment, valid_from, valid_to
    FROM hw2_customer_features_offline
    WHERE customer_id = ?
      AND valid_from <= ?
      AND (valid_to > ? OR valid_to IS NULL)
    """,
    (
        42,
        "2024-12-20 10:00:00+00:00",
        "2024-12-20 10:00:00+00:00"
    )
)


show_query(
    "AS-OF SELECT №3: client 1 on 20.11.2024",
    """
    SELECT customer_id, orders_30d, avg_order_value,
           segment, valid_from, valid_to
    FROM hw2_customer_features_offline
    WHERE customer_id = ?
      AND valid_from <= ?
      AND (valid_to > ? OR valid_to IS NULL)
    """,
    (
        1,
        "2024-11-20 10:00:00+00:00",
        "2024-11-20 10:00:00+00:00"
    )
)


training_query = """
SELECT
    e.event_id,
    e.customer_id,
    e.prediction_ts,
    e.label_horizon_end,
    e.churn_in_horizon,
    f.orders_30d,
    f.avg_order_value,
    f.segment,
    f.valid_from,
    f.valid_to
FROM hw2_training_events AS e
LEFT JOIN hw2_customer_features_offline AS f
    ON f.customer_id = e.customer_id
   AND f.valid_from <= e.prediction_ts
   AND (f.valid_to > e.prediction_ts OR f.valid_to IS NULL)
ORDER BY e.event_id
"""

training_df = show_query(
    "AS-OF JOIN: temporally correct training data",
    training_query
)


overlap_query = """
SELECT
    a.customer_id,
    a.valid_from AS a_from,
    a.valid_to AS a_to,
    b.valid_from AS b_from,
    b.valid_to AS b_to
FROM hw2_customer_features_offline AS a
JOIN hw2_customer_features_offline AS b
    ON a.customer_id = b.customer_id
   AND a.valid_from <
       COALESCE(b.valid_to, '9999-12-31 23:59:59+00:00')
   AND b.valid_from <
       COALESCE(a.valid_to, '9999-12-31 23:59:59+00:00')
   AND a.valid_from < b.valid_from
"""

overlaps = show_query(
    "Check for overlapping intervals (must be 0 rows)",
    overlap_query
)

print("Number of overlapping intervals found:", len(overlaps))


wrong_query = """
SELECT
    e.event_id,
    e.customer_id,
    e.prediction_ts,
    f.orders_30d,
    f.avg_order_value,
    f.segment,
    f.valid_from
FROM hw2_training_events AS e
JOIN hw2_customer_features_offline AS f
    ON f.customer_id = e.customer_id
ORDER BY e.event_id, f.valid_from
"""

wrong_df = show_query(
    "WRONG JOIN: all historical versions",
    wrong_query
)

print("Rows after AS-OF JOIN:", len(training_df))
print("Rows after wrong JOIN:", len(wrong_df))


show_query(
    "Number of versions for each customer",
    """
    SELECT customer_id,
           COUNT(*) AS versions_count,
           MIN(valid_from) AS first_version,
           MAX(valid_from) AS latest_version
    FROM hw2_customer_features_offline
    GROUP BY customer_id
    ORDER BY customer_id
    """
)

show_query(
    "Total number of historical versions",
    "SELECT COUNT(*) AS total_feature_versions "
    "FROM hw2_customer_features_offline"
)

show_query(
    "Total number of training events",
    "SELECT COUNT(*) AS total_training_events "
    "FROM hw2_training_events"
)


assert len(features) == 12
assert len(events) == 10
assert len(overlaps) == 0
assert len(training_df) == len(events)
assert training_df["orders_30d"].notna().all()

print("\nAll checks passed successfully!")
print("3 clients, 12 historical versions, 10 training events.")
print("AS-OF JOIN did not create duplicates.")
print("No overlapping temporal intervals found.")

Tables created.
Added historical versions: 12
Educational events added: 10

AS-OF SELECT №1: client 42 on 10.12.2024


,customer_id,orders_30d,avg_order_value,segment,valid_from,valid_to
0,42,2,100.0,A,2024-11-01 00:00:00+00:00,2024-12-15 00:00:00+00:00



AS-OF SELECT №2: client 42 on 20.12.2024


,customer_id,orders_30d,avg_order_value,segment,valid_from,valid_to
0,42,5,200.0,B,2024-12-15 00:00:00+00:00,2025-01-15 00:00:00+00:00



AS-OF SELECT №3: client 1 on 20.11.2024


...


Total number of historical versions


,total_feature_versions
0,12



Total number of training events


,total_training_events
0,10



All checks passed successfully!
3 clients, 12 historical versions, 10 training events.
AS-OF JOIN did not create duplicates.
No overlapping temporal intervals found.


Простий "JOIN" лише за "customer_id" приєднав би до кожної навчальної події всі історичні версії ознак клієнта. Це створило б дублікати навчальних прикладів і могло б включити майбутні значення, яких не існувало на момент прогнозування. Така проблема називається витоком даних "data leakage" і може призвести до завищеної оцінки якості моделі. Часовий "AS-OF JOIN" усуває цю проблему, забезпечуючи відповідність ознак конкретному моменту прогнозування.

Завдання 5.

1. Що було найскладнішим.

Найскладнішим етапом стала реалізація "point-in-time correctness", оскільки необхідно гарантувати, що модель використовує лише ті ознаки, які були доступні на момент прогнозування. Для цього потрібно правильно визначити часові межі "valid_from" та "valid_to", уникнути перекривання версій та побудувати "AS-OF JOIN" за "prediction_ts". Вибір сутностей і визначення кардинальностей також були важливими, які можна перевірити за допомогою первинних та зовнішніх ключів.

2. Проєктні компроміси.

Основні сутності, клієнтів, історичні ознаки та навчальні події, збережено в окремих таблицях. Така нормалізація зменшує дублювання даних і спрощує їх оновлення. Зв’язки між таблицями реалізовано через зовнішні ключі. Історію ознак зберігаємо окремими часовими версіями, щоб відтворювати стан даних у минулому. Для online-рівня можна використати денормалізоване представлення, яке об'єднує потрібні ознаки в один запис. Це прискорює читання, хоча потребує додаткового контролю узгодженості під час оновлень.

3. Як прискорити online lookup.

За великої кількості користувачів доцільно зберігати попередньо сформований денормалізований рядок ознак за "entity_id". Попередня матеріалізація зменшить кількість операцій читання та об'єднання таблиць. Кешування допоможе швидко повертати часто запитувані ознаки. Для збільшення пропускної здатності можна застосувати горизонтальне масштабування. Водночас необхідно контролювати свіжість даних і визначити допустиму затримку оновлення.

4. Ризики еволюції схеми протягом шести місяців.

Основними ризиками є перейменування ознак, зміна типів даних, видалення колонок і зміна семантики "feature". Такі зміни можуть порушити сумісність із наявними моделями та процесами формування навчальних даних. Для зменшення ризиків потрібно версіонувати схеми й ознаки, документувати зміни та перевіряти сумісність перед розгортанням. Старі версії моделей мають зберігати доступ до відповідного набору ознак, щоб результати прогнозування залишалися відтворюваними.